# 01 — Ingestion dans le data lake : couche *bronze*

Consommation électrique Enedis (1/2 h, par région, 2022) et température quotidienne départementale (ODRÉ).
Les données brutes sont déposées **telles quelles** dans RustFS, au format Delta.
Seules modifications autorisées : les noms de colonnes (caractères interdits par Delta) et deux colonnes techniques
(`_source`, `_ingere_le`).

```
data/raw/enedis/*.csv       ──►  s3a://datalake/bronze/enedis_conso
data/raw/meteo/*.csv        ──►  s3a://datalake/bronze/meteo_departements
ref/departements_regions.csv ──►  s3a://datalake/bronze/ref_departements
```

Prérequis : `bash scripts/telecharger_donnees.sh` puis `docker compose up -d`.

In [1]:
import sys; sys.path.append("/home/jovyan/work")
import glob, re, unicodedata
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, creer_bucket, format_tables, RAW, LAKE, WORK

creer_bucket("datalake")

Bucket « datalake » déjà présent.


In [2]:
spark = creer_session("bronze")
FMT = format_tables(spark)
FMT

Préparation de Spark 4.2.0 (Hadoop 3.5.0) + Delta Lake + data lake S3 … le premier démarrage peut prendre quelques minutes.
Session prête. Spark UI : http://localhost:4040


'delta'

## 1. Fonctions utilitaires

In [3]:
def detecter_sep(chemin):
    """Les CSV français utilisent souvent « ; » : on le détecte sur la première ligne."""
    with open(chemin, encoding="utf-8-sig") as f:
        ligne = f.readline()
    return ";" if ligne.count(";") >= ligne.count(",") else ","


def nettoyer_noms(df):
    """Delta refuse certains caractères dans les noms de colonnes : on ne renomme que ceux-là."""
    for c in df.columns:
        propre = unicodedata.normalize("NFKD", c.strip().lower()).encode("ascii", "ignore").decode()
        propre = re.sub(r"[ ,;{}()\n\t=]+", "_", propre)
        if propre != c:
            df = df.withColumnRenamed(c, propre)
    return df


def vers_bronze(df, nom):
    """Écrit en bronze avec traçabilité (fichier source, date d'ingestion). Aucune autre transformation."""
    df = (nettoyer_noms(df)
          .withColumn("_source", F.col("_metadata.file_name"))
          .withColumn("_ingere_le", F.current_timestamp()))
    df.write.format(FMT).mode("overwrite").save(f"{LAKE}/bronze/{nom}")
    relu = spark.read.format(FMT).load(f"{LAKE}/bronze/{nom}")
    print(f"bronze/{nom} : {relu.count():,} lignes, {len(relu.columns)} colonnes")
    return relu

## 2. Consommation Enedis → bronze

Tout est lu en **texte** (`inferSchema=False`) : le typage est le travail de la couche *silver*.

In [4]:
fichiers = sorted(glob.glob(f"{RAW}/enedis/*.csv"))
assert fichiers, "Aucun CSV Enedis : lancez d'abord  bash scripts/telecharger_donnees.sh"
sep = detecter_sep(fichiers[0])
print(len(fichiers), "fichiers, séparateur", repr(sep))

conso_raw = spark.read.csv(fichiers, header=True, sep=sep, inferSchema=False)
conso = vers_bronze(conso_raw, "enedis_conso")
conso.printSchema()
conso.show(5, truncate=False)

4 fichiers, séparateur ';'
bronze/enedis_conso : 17,239,680 lignes, 17 colonnes
root
 |-- horodate: string (nullable = true)
 |-- region: string (nullable = true)
 |-- code_region: string (nullable = true)
 |-- profil: string (nullable = true)
 |-- plage_de_puissance_souscrite: string (nullable = true)
 |-- nb_points_soutirage: string (nullable = true)
 |-- total_energie_soutiree_wh_: string (nullable = true)
 |-- courbe_moyenne_n1_wh_: string (nullable = true)
 |-- indice_representativite_courbe_n1_%_: string (nullable = true)
 |-- courbe_moyenne_n2_wh_: string (nullable = true)
 |-- indice_representativite_courbe_n2_%_: string (nullable = true)
 |-- courbe_moyenne_n1_+_n2_wh_: string (nullable = true)
 |-- indice_representativite_courbe_n1_+_n2_%_: string (nullable = true)
 |-- jour_max_du_mois_0/1_: string (nullable = true)
 |-- semaine_max_du_mois_0/1_: string (nullable = true)
 |-- _source: string (nullable = true)
 |-- _ingere_le: timestamp (nullable = true)

+-------------------

In [5]:
# Contrôles : volume attendu (> 2 millions de lignes) et répartition par fichier source
n = conso.count()
assert n > 2_000_000, f"Seulement {n:,} lignes : vérifier le téléchargement"
conso.groupBy("_source").count().orderBy("_source").show(truncate=False)

+---------------------------+-------+
|_source                    |count  |
+---------------------------+-------+
|conso_region_2022_part1.csv|4347312|
|conso_region_2022_part2.csv|4345344|
|conso_region_2022_part3.csv|4298112|
|conso_region_2022_part4.csv|4248912|
+---------------------------+-------+



## 3. Températures ODRÉ → bronze

In [6]:
fichier_meteo = f"{RAW}/meteo/temperature_departementale.csv"
meteo_raw = spark.read.csv(fichier_meteo, header=True, sep=detecter_sep(fichier_meteo), inferSchema=False)
meteo = vers_bronze(meteo_raw, "meteo_departements")
meteo.printSchema()
meteo.show(5, truncate=False)

bronze/meteo_departements : 309,696 lignes, 8 colonnes
root
 |-- date: string (nullable = true)
 |-- code_insee_departement: string (nullable = true)
 |-- departement: string (nullable = true)
 |-- tmin_c_: string (nullable = true)
 |-- tmax_c_: string (nullable = true)
 |-- tmoy_c_: string (nullable = true)
 |-- _source: string (nullable = true)
 |-- _ingere_le: timestamp (nullable = true)

+----------+----------------------+--------------+-------+-------+-------+------------------------------+--------------------------+
|date      |code_insee_departement|departement   |tmin_c_|tmax_c_|tmoy_c_|_source                       |_ingere_le                |
+----------+----------------------+--------------+-------+-------+-------+------------------------------+--------------------------+
|2020-12-14|01                    |Ain           |-1.2   |12.6   |5.7    |temperature_departementale.csv|2026-11-26 20:38:24.452517|
|2020-12-14|95                    |Val-d'Oise    |7.65   |11.77  |9.71   

## 4. Référentiel départements → régions

Petit fichier statique versionné dans `ref/` (96 départements métropolitains) : il permet de passer de la maille
départementale (météo) à la maille régionale (Enedis) en couche *silver*.

In [7]:
ref_raw = spark.read.csv(f"{WORK}/ref/departements_regions.csv", header=True, sep=";", inferSchema=False)
ref = vers_bronze(ref_raw, "ref_departements")
ref.show(5)

bronze/ref_departements : 96 lignes, 5 colonnes
+----------------+-----------+--------------------+--------------------+--------------------+
|code_departement|code_region|          nom_region|             _source|          _ingere_le|
+----------------+-----------+--------------------+--------------------+--------------------+
|              01|         84|Auvergne-Rhône-Alpes|departements_regi...|2026-11-26 20:38:...|
|              02|         32|     Hauts-de-France|departements_regi...|2026-11-26 20:38:...|
|              03|         84|Auvergne-Rhône-Alpes|departements_regi...|2026-11-26 20:38:...|
|              04|         93|Provence-Alpes-Cô...|departements_regi...|2026-11-26 20:38:...|
|              05|         93|Provence-Alpes-Cô...|departements_regi...|2026-11-26 20:38:...|
+----------------+-----------+--------------------+--------------------+--------------------+
only showing top 5 rows


## 5. Historique Delta

In [8]:
if FMT == "delta":
    from delta.tables import DeltaTable
    for t in ["enedis_conso", "meteo_departements", "ref_departements"]:
        print(t)
        DeltaTable.forPath(spark, f"{LAKE}/bronze/{t}").history().select("version", "timestamp", "operation").show(truncate=False)

enedis_conso
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-11-26 20:38:03|WRITE    |
|0      |2026-11-20 22:21:55|WRITE    |
+-------+-------------------+---------+

meteo_departements
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-11-26 20:38:28|WRITE    |
|0      |2026-11-20 22:22:41|WRITE    |
+-------+-------------------+---------+

ref_departements
+-------+-------------------+---------+
|version|timestamp          |operation|
+-------+-------------------+---------+
|1      |2026-11-26 20:38:41|WRITE    |
|0      |2026-11-20 22:22:53|WRITE    |
+-------+-------------------+---------+



## C'est réussi quand…
- la console RustFS (http://localhost:9001) montre `bronze/enedis_conso`, `bronze/meteo_departements`, `bronze/ref_departements` ;
- `bronze/enedis_conso` dépasse 2 millions de lignes ;
- les noms de colonnes des deux jeux sont notés pour écrire les règles *silver* (notebook 02).

In [9]:
spark.stop()